# 02 — Classical ML demo (frame-level + sequence-level)

Trains LogisticRegression and RandomForest on **both** the frame-level task
(per 10 ms frame) and the sequence-level task (per 500 ms window). This
matches the comparison the original notebook lacked.

In [ ]:
import json, pickle
from prosody.data import load_corpus, speaker_split
from prosody._train_classical import (
    train_frame_level_classical,
    train_sequence_level_classical,
)
from prosody.evaluate import format_metrics

In [ ]:
# Cached corpus for speed (generated by `make neural` or scripts/run_neural.py)
with open("artifacts/corpus.pkl", "rb") as f:
    corpus = pickle.load(f)
sp = speaker_split(corpus, val_speakers=["m2b"], test_speakers=["f3a"])

## Frame-level (per 10 ms)

In [ ]:
frame_results = train_frame_level_classical(sp.train, sp.val, sp.test)
for task, models in frame_results.items():
    for name, r in models.items():
        print(f"  [{task:>10s}] {name:<22s} {format_metrics(r.test)}")

## Sequence-level (per 500 ms window)

In [ ]:
seq_results = train_sequence_level_classical(sp.train, sp.val, sp.test)
for task, models in seq_results.items():
    for name, r in models.items():
        print(f"  [{task:>10s}] {name:<22s} {format_metrics(r.test)}")

## Read off the trivial baseline

Always-predict-positive F1 = 2p / (1+p) where p = positive rate. If a model
F1 is at or below this, it is no better than predicting every frame positive.

In [ ]:
for task, models in seq_results.items():
    any_r = next(iter(models.values()))
    p = any_r.test.positive_rate
    triv = (2*p) / (1+p)
    print(f"sequence-level {task}: trivial F1 = {triv:.3f} (positive rate {p:.3f})")